# Restvolumen je Projekt

Schritt 1 aus Spec Abschnitt 10: `/v4/projects` und `/v2/entrygroups` abfragen und
das Restvolumen je Projekt berechnen (Abschnitt 5.1) – als Zwischenschritt **vor**
der Monte-Carlo-Logik.

Das Notebook ruft nur ab und stellt dar; API-Zugriff und Rechenlogik liegen im Paket
`umsatzprognose`. Warum die Endpunkte so aufgerufen werden, wie sie hier aufgerufen
werden, steht im Docstring von `umsatzprognose.api` – geprüft per `curl` gegen die
echte Installation, weil `docs.clockodo.com` als JavaScript-Anwendung nicht auslesbar
war.

In [ ]:
# Nur in Google Colab: Projekt aus GitHub installieren, weil das lokale venv dort
# nicht zur Verfuegung steht. Lokal passiert hier nichts, dort liefert `uv sync` die
# Umgebung. Das Repository ist oeffentlich, deshalb braucht pip kein Token.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Fuer einen reproduzierbaren Lauf auf einen Tag setzen statt auf "main".
PAKET_REF = "main"

if IN_COLAB:
    !pip install --quiet "git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"

IN_COLAB

In [ ]:
from umsatzprognose.api import ClockodoClient
from umsatzprognose.auftragsvolumen import budgets_je_projekt
from umsatzprognose.config import load_credentials_auto
from umsatzprognose.restvolumen import restvolumen_je_projekt, summe_prognosewirksam
from umsatzprognose.tabellen import restvolumen_tabelle
from umsatzprognose.verbrauchtes_volumen import revenue_je_projekt

# Colab-Secrets in Colab, lokal die .env - die Unterscheidung trifft das Paket.
creds = load_credentials_auto()
client = ClockodoClient(creds)
print("Angemeldet als", creds.api_user)

## Auftragsvolumen aus `/v4/projects`

In [ ]:
# ENTSCHEIDEN: Von 895 Projekten sind nur 122 aktiv, der Rest ist abgeschlossen oder
# archiviert. Fuer eine Prognose kuenftiger Umsaetze zaehlen laufende Projekte. Die
# Spec sagt dazu nichts - die Abgrenzung ist eine Annahme und gehoert bestaetigt oder
# korrigiert.
NUR_AKTIVE = True

projects, paging = client.projects()
auszug = budgets_je_projekt(projects, nur_aktive=NUR_AKTIVE)
budgets = auszug.budgets

print(f"{len(projects)} Projekte geladen (paging: {paging})")
print(
    f"{len(budgets)} beruecksichtigt (NUR_AKTIVE={NUR_AKTIVE}), davon "
    f"{sum(v is None for v in budgets.values())} ohne verwertbares Euro-Gesamtbudget"
)

# Sonderformen, die "budget.amount" seine Euro-Bedeutung nehmen: Stundenbudget
# (monetary=false), Budget je Intervall, Budget aus Teilprojekten. Bei den aktiven
# Projekten bisher leer; die Ausgabe macht sichtbar, wenn das kippt.
if auszug.unbenutzbar:
    print(f"  Budget nicht als Euro-Gesamtbudget lesbar: {auszug.unbenutzbar}")

## Verbrauchtes Volumen aus `/v2/entrygroups`

Gruppierung nach Projekt über die gesamte Projekthistorie – `revenue_kumuliert` in
Abschnitt 5.1 ist der Gesamtverbrauch, nicht der eines Monats.

In [ ]:
groups = client.entrygroups_je_projekt()
revenue_kumuliert, ohne_projekt = revenue_je_projekt(groups)

print(f"{len(groups)} Gruppen geladen, Verbrauch fuer {len(revenue_kumuliert)} Projekte")
print(
    f"Gruppen ohne Projektbezug (group == 0): {len(ohne_projekt)}, "
    f"Umsatz {sum(float(g.get('revenue') or 0) for g in ohne_projekt):,.2f} EUR"
)

## Restvolumen (Spec 5.1)

`roh` ist `budget.amount - revenue_kumuliert` und kann negativ sein, weil
`budget.hard` false ist. `prognosewirksam` kappt bei 0 – nur dieser Teil kann noch
abgerufen werden und geht in die Simulation ein.

Laut Spec 5.1 (seit v0.5) kann eine Budgetüberschreitung **nur historisch** entstehen;
die Prognose überschreitet das Budget nicht. Für Projekte mit `ueberschritten == True`
wird deshalb kein zukünftiger Umsatz prognostiziert.

In [ ]:
restvolumina, ohne_budget = restvolumen_je_projekt(budgets, revenue_kumuliert)
tabelle = restvolumen_tabelle(restvolumina)

print(f"Prognosewirksames Restvolumen gesamt: {summe_prognosewirksam(restvolumina):,.2f} EUR")
print(f"Projekte mit Budgetueberschreitung:   {int(tabelle['ueberschritten'].sum())}")
print(f"Projekte ohne Budget (ausgeschlossen): {len(ohne_budget)}, z. B. {ohne_budget[:5]}")

tabelle.head(30)

## Offen

**ENTSCHEIDEN – 78 der 122 aktiven Projekte haben kein Budget** und fallen damit aus
der Prognose. Ein Blick auf die Namen zeigt, was das überwiegend ist: Schulungs- und
Ausbildungsprodukte (`A-CSM`, `A-CSPO`, `ACC`, `Agile Change Management`, …), also
Katalogpositionen ohne beauftragtes Volumen. Genau das rechnet die Spec dem
Kurzfristgeschäft zu und schließt es aus dem MVP aus. Zu prüfen bleibt, ob unter den 78
auch echte Bestandsprojekte stecken, bei denen nur das Budget fehlt – dann ist es ein
Pflegethema, kein Modellthema.

**ENTSCHEIDEN – zwei aktive Projekte sind `completed`,** eines davon mit 12.424 EUR
offenem Budget. Sie gehen derzeit in die Prognose ein, obwohl sie fachlich beendet sein
dürften. Das Feld kennt die Spec nicht.

**Effektiver Stundensatz:** `hourly_rate` aus `/v2/entrygroups` taugt dafür nicht – es
ist genau dann gesetzt, wenn das Projekt einen einheitlichen Satz und keine
Pauschalleistungen hat, also bei 92 von 870 Gruppen, und dort meist 0. Für die
restlichen 778 ist es `null`. Der Satz muss aus `revenue` und `duration` abgeleitet
werden; die Definition steht in Spec v0.3, die nicht im Repository liegt. Damit fehlt
auch die Normalisierung von Pauschalleistungen (5.1) – 8 Gruppen haben Umsatz ohne jede
erfasste Zeit.

**Nächster Schritt** laut Abschnitt 10: Abrufquoten-Verteilungen je Referenzklasse aus
der `entrygroups`-Historie schätzen.